<a href="https://colab.research.google.com/github/riyaindap7/BE_IT/blob/main/optimized_fine_tuning_qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================
# STEP 0: INSTALL DEPENDENCIES
# =====================================================
!pip install --upgrade transformers accelerate "datasets>=2.19.0,<3.0.0" peft bitsandbytes torch torchvision sentencepiece \
               nltk rouge-score sacrebleu tree-sitter codebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 136.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 11.1 MB/s eta 0:00:00
   ━━

In [ ]:
# =====================================================
# STEP 1: IMPORTS
# =====================================================
import torch
import nltk
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, PeftModel
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from codebleu import calc_codebleu

nltk.download("wordnet")
nltk.download("omw-1.4")

# =====================================================
# STEP 2: LOAD DATASETS (TRAIN + TEST)
# =====================================================
LANGUAGES = ["python", "java", "javascript", "php"]

TRAIN_SPLIT = "train[:1%]"
TEST_SPLIT = "test[:1000]"

train_sets = []
test_sets = []

for lang in LANGUAGES:
    train_ds = load_dataset(
        "code_search_net",
        name=lang,
        split=TRAIN_SPLIT,
        trust_remote_code=True
    )

    test_ds = load_dataset(
        "code_search_net",
        name=lang,
        split=TEST_SPLIT,
        trust_remote_code=True
    )

    # The 'language' column is already present in the dataset
    # when loaded with trust_remote_code=True, so these lines are redundant.
    # train_ds = train_ds.add_column("language", [lang] * len(train_ds))
    # test_ds = test_ds.add_column("language", [lang] * len(test_ds))

    train_sets.append(train_ds)
    test_sets.append(test_ds)

train_dataset = concatenate_datasets(train_sets)
test_dataset = concatenate_datasets(test_sets)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

# =====================================================
# STEP 3: FORMAT DATA (INSTRUCTION STYLE)
# =====================================================
def format_example(example):
    instruction = (
        f"Generate a concise and accurate documentation comment "
        f"for the following {example['language']} code."
    )

    prompt = f"""### Instruction:
{instruction}

### Code:
{example['func_code_string']}

### Docstring:
"""

    return {
        "prompt": prompt,
        "reference": example["func_documentation_string"]
    }

train_dataset = train_dataset.map(
    format_example,
    remove_columns=train_dataset.column_names
)

test_dataset = test_dataset.map(
    format_example,
    remove_columns=test_dataset.column_names
)

# =====================================================
# STEP 4: LOAD MODEL (QLoRA)
# =====================================================
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# =====================================================
# STEP 5: APPLY LoRA
# =====================================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.gradient_checkpointing_enable()
model.config.use_cache = False

# =====================================================
# STEP 6: TOKENIZATION
# =====================================================
MAX_LEN = 384

def tokenize_train(example):
    text = example["prompt"] + example["reference"]
    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_tokenized = train_dataset.map(
    tokenize_train,
    batched=True,
    remove_columns=train_dataset.column_names
)

# =====================================================
# STEP 7: TRAINING
# =====================================================
training_args = TrainingArguments(
    output_dir="./writer_lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized
)

trainer.train()

model.save_pretrained("./content/drive/MyDrive/models/writer_lora")
tokenizer.save_pretrained("./content/drive/MyDrive/models/writer_lora")

# =====================================================
# STEP 8: LOAD MODEL FOR EVALUATION
# =====================================================
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

writer_model = PeftModel.from_pretrained(
    base_model,
    "./writer_lora"
)
writer_model.eval()

# =====================================================
# STEP 9: GENERATE PREDICTIONS
# =====================================================
def generate_doc(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = writer_model.generate(
            **inputs,
            max_new_tokens=120
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

predictions = []
references = []
codes = []

for sample in test_dataset:
    pred = generate_doc(sample["prompt"])
    predictions.append(pred)
    references.append(sample["reference"])
    codes.append(sample["prompt"])

# =====================================================
# STEP 10: EVALUATION METRICS
# =====================================================

# BLEU
bleu_score = corpus_bleu(
    [[ref.split()] for ref in references],
    [pred.split() for pred in predictions]
)

# METEOR
meteor = sum(
    meteor_score([ref], pred)
    for ref, pred in zip(references, predictions)
) / len(predictions)

# ROUGE-L
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rouge_l = sum(
    scorer.score(ref, pred)["rougeL"].fmeasure
    for ref, pred in zip(references, predictions)
) / len(predictions)

# CodeBLEU
codebleu_score = calc_codebleu(
    references,
    predictions,
    lang="python"  # dominant evaluation language
)["codebleu"]

# =====================================================
# STEP 11: PRINT RESULTS
# =====================================================
print("\n========== EVALUATION RESULTS ==========")
print(f"BLEU      : {bleu_score:.4f}")
print(f"METEOR    : {meteor:.4f}")
print(f"ROUGE-L   : {rouge_l:.4f}")
print(f"CodeBLEU  : {codebleu_score:.4f}")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/454451 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


Map:   0%|          | 0/15143 [00:00<?, ? examples/s]

Step,Training Loss
50,1.102457
100,0.381280
150,0.341112
200,0.412885
250,0.361548
300,0.337176
350,0.326934
400,0.361010
450,0.314814
500,0.328243
